## Fabric_Lakehouse_DisasterRecovery_Copy_Tables

Copies Bronze/Silver tables from source lakehouse to backup lakehouse using Delta-to-Delta replication via ABFSS paths. Supports full backup mode or test mode (5 tables). Includes incremental backup based on modification time.

**Global Settings:**
- `BACKUP_MODE`: "all" or "test" (test limits to 5 tables)
- `INCREMENTAL_MODE`: True = only backup tables where source is newer than backup
- `OVERWRITE_EXISTING`: True = replace existing backup tables
- `VALIDATE_AFTER_BACKUP`: True = compare row counts source vs backup
- `FAIL_ON_MISMATCH`: False = continue on validation failures, True = stop


### Import Libraries

In [ ]:
import sempy.fabric as fabric
import sempy_labs.lakehouse as lake
from datetime import datetime
import pandas as pd

### Create Parameters

In [ ]:
# Backup Mode - controls which tables to backup
BACKUP_MODE = "all"  # Options: "all", "test"

# Test mode - when BACKUP_MODE = "test"
TEST_TABLE_COUNT = 5  # Number of tables to use for testing (selected from Head)

# Incremental mode - Saves a full backup as selective based on changed Delta Logs
INCREMENTAL_MODE = True  # If True, only backup tables where source lastModified > backup lastModified

# Backup behavior
OVERWRITE_EXISTING = True  # True = overwrite, False = skip existing tables
SAVE_METADATA = True  # Save table metadata snapshot to backup lakehouse

# Validation - inline at backup time to avoid source drift false positives
FAIL_ON_MISMATCH = False  # If True, stop execution if any table row count doesn't match at backup time


### Create Functions

In [ ]:
# Check if table needs backup by comparing source vs backup lastModified timestamps
def needs_backup(table_name, source_ws_id, source_lh_id, backup_ws_id, backup_lh_id):
    """Check if table needs backup by comparing modification times"""
    try:
        source_path = f"abfss://{source_ws_id}@onelake.dfs.fabric.microsoft.com/{source_lh_id}/Tables/{table_name}"
        backup_path = f"abfss://{backup_ws_id}@onelake.dfs.fabric.microsoft.com/{backup_lh_id}/Tables/{table_name}"
        
        source_detail = spark.sql(f"DESCRIBE DETAIL delta.`{source_path}`").collect()[0]
        
        try:
            backup_detail = spark.sql(f"DESCRIBE DETAIL delta.`{backup_path}`").collect()[0]
            return source_detail['lastModified'] > backup_detail['lastModified']
        except:
            return True  # Backup doesn't exist - needs backup
            
    except Exception as e:
        print(f"Error checking {table_name}: {e}")
        return True  # Default to backing up on error


# Copy table from source to backup, validates row count immediately after write
def backup_table(table_name, source_ws_id, source_lh_id, backup_ws_id, backup_lh_id):
    """Copy table from source to backup using Delta-to-Delta, validates immediately after write"""
    try:
        source_path = f"abfss://{source_ws_id}@onelake.dfs.fabric.microsoft.com/{source_lh_id}/Tables/{table_name}"
        df = spark.read.format("delta").load(source_path)
        source_rows = df.count()
        
        backup_path = f"abfss://{backup_ws_id}@onelake.dfs.fabric.microsoft.com/{backup_lh_id}/Tables/{table_name}"
        df.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(backup_path)
        
        # Validate immediately - backup was just written, source df still in memory
        backup_rows = spark.read.format("delta").load(backup_path).count()
        match = source_rows == backup_rows
        
        return {
            'table_name': table_name,
            'source_rows': source_rows,
            'backup_rows': backup_rows,
            'match': match,
            'status': 'success' if match else 'mismatch',
            'error': None
        }
        
    except Exception as e:
        return {
            'table_name': table_name,
            'source_rows': 0,
            'backup_rows': 0,
            'match': False,
            'status': 'failed',
            'error': str(e)
        }


### Load workspace Variables

In [ ]:
# Load workspace-specific configuration
variable_lib = notebookutils.variableLibrary.getLibrary("Workspace_Variables")

# Source workspace (current environment - DEV/UAT/PROD)
source_workspace_id = variable_lib.getVariable("p_Metadata_WorkspaceID")
source_lakehouse_id = variable_lib.getVariable("p_Contoso_Lakehouse_ObjectID")
source_lakehouse_name = variable_lib.getVariable("p_Contoso_Lakehouse_Name")

# Get all workspaces
all_workspaces = fabric.list_workspaces()

# Backup workspace
backup_workspace_id = all_workspaces[all_workspaces['Name'] == 'Contoso Fabric Lakehouse (BACKUP)']['Id'].iloc[0]

# Get backup lakehouse ID
backup_lakehouses = fabric.list_items(item_type="Lakehouse", workspace=backup_workspace_id)
backup_lakehouse_id = backup_lakehouses[backup_lakehouses['Display Name'] == 'Contoso_Fabric_Lakehouse_Backup']['Id'].iloc[0]
backup_lakehouse_name = 'Contoso_Fabric_Lakehouse_Backup'

# Print configuration
print("Backup Configuration:")
print("=" * 80)
print(f"Source Workspace:  {source_workspace_id}")
print(f"Source Lakehouse:  {source_lakehouse_name} ({source_lakehouse_id})")
print(f"Backup Workspace:  {backup_workspace_id}")
print(f"Backup Lakehouse:  {backup_lakehouse_name} ({backup_lakehouse_id})")
print(f"Backup Mode:       {BACKUP_MODE}")
print("=" * 80)

### Get Source Tables

In [ ]:
# Get all tables from source lakehouse
source_tables_df = lake.get_lakehouse_tables(
    lakehouse=source_lakehouse_name,
    workspace=source_workspace_id
)

print(f"Total tables in source lakehouse: {len(source_tables_df)}")
display(source_tables_df.head(10))

### Select Tables

In [ ]:
# Determine which tables to backup based on BACKUP_MODE
if BACKUP_MODE == "test":
    tables_to_backup = source_tables_df['Table Name'].head(TEST_TABLE_COUNT).tolist()
    print(f"TEST MODE: Backing up first {TEST_TABLE_COUNT} tables")
else:
    tables_to_backup = source_tables_df['Table Name'].tolist()
    print(f"ALL MODE: Backing up all {len(tables_to_backup)} tables")

# Display selected tables
print(f"\nTables selected for backup: {len(tables_to_backup)}")
print(tables_to_backup[:10] if len(tables_to_backup) > 10 else tables_to_backup)
if len(tables_to_backup) > 10:
    print(f"... and {len(tables_to_backup) - 10} more")

## Save Metadata

In [ ]:
if SAVE_METADATA:
    print("Saving metadata snapshot...")
    
    # Add snapshot metadata
    metadata_df = source_tables_df.copy()
    metadata_df['snapshot_date'] = datetime.now()
    metadata_df['source_workspace'] = source_workspace_id
    metadata_df['source_lakehouse'] = source_lakehouse_id
    
    # Clean column names for Delta compatibility
    metadata_df.columns = metadata_df.columns.str.replace(' ', '_')
    
    # Convert to Spark and write
    spark_df = spark.createDataFrame(metadata_df)
    backup_path = f"abfss://{backup_workspace_id}@onelake.dfs.fabric.microsoft.com/{backup_lakehouse_id}/Tables/lakehouse_table_metadata"
    spark_df.write.mode("append").format("delta").option("mergeSchema", "true").save(backup_path)
    
    print(f"✓ Saved metadata for {len(source_tables_df)} tables")

### Execute Backup

In [ ]:
print(f"\nStarting backup of {len(tables_to_backup)} tables...")
print("=" * 80)

backup_results = []
start_time = datetime.now()

for i, table_name in enumerate(tables_to_backup, 1):

    # Check if backup needed (incremental mode)
    if INCREMENTAL_MODE:
        if not needs_backup(table_name, source_workspace_id, source_lakehouse_id, backup_workspace_id, backup_lakehouse_id):
            print(f"[{i}/{len(tables_to_backup)}] ⊘ {table_name}: No changes, skipped")
            continue

    result = backup_table(
        table_name,
        source_workspace_id,
        source_lakehouse_id,
        backup_workspace_id,
        backup_lakehouse_id
    )

    backup_results.append(result)

    # Progress output
    if result['status'] == 'success':
        print(f"[{i}/{len(tables_to_backup)}] ✓ {table_name}: {result['source_rows']:,} rows")
    elif result['status'] == 'mismatch':
        print(f"[{i}/{len(tables_to_backup)}] ✗ {table_name}: Source={result['source_rows']:,} | Backup={result['backup_rows']:,}")
    else:
        print(f"[{i}/{len(tables_to_backup)}] ✗ {table_name}: {result['error']}")

    # Stop on mismatch if configured
    if FAIL_ON_MISMATCH and not result['match']:
        print(f"\nStopping - FAIL_ON_MISMATCH=True and mismatch detected on {table_name}")
        break

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

# Summary
success_count = sum(1 for r in backup_results if r['status'] == 'success')
mismatch_count = sum(1 for r in backup_results if r['status'] == 'mismatch')
failed_count = sum(1 for r in backup_results if r['status'] == 'failed')
total_rows = sum(r['source_rows'] for r in backup_results)

print("\n" + "=" * 80)
print("BACKUP SUMMARY")
print("=" * 80)
print(f"Total Tables:    {len(backup_results)}")
print(f"Matched:         {success_count}")
print(f"Mismatched:      {mismatch_count}")
print(f"Failed:          {failed_count}")
print(f"Total Rows:      {total_rows:,}")
print(f"Duration:        {duration:.1f} seconds")
print("=" * 80)

# Display full results
results_pdf = pd.DataFrame(backup_results)
results_pdf['error'] = results_pdf['error'].fillna('')
display(results_pdf)


### Validation Results

Validation is performed inline at backup time — source rows are captured before the write, backup rows counted immediately after. This eliminates false positives from source data changing during backup.


In [ ]:
# Show any mismatches from inline validation
mismatches = [r for r in backup_results if r['status'] == 'mismatch']
failures = [r for r in backup_results if r['status'] == 'failed']

print(f"Matched:    {sum(1 for r in backup_results if r['status'] == 'success')}")
print(f"Mismatched: {len(mismatches)}")
print(f"Failed:     {len(failures)}")

if mismatches:
    print("\nMismatch detail:")
    display(pd.DataFrame(mismatches)[['table_name', 'source_rows', 'backup_rows']])

if failures:
    print("\nFailed tables:")
    display(pd.DataFrame(failures)[['table_name', 'error']])

if not mismatches and not failures:
    print("\n✓ All tables backed up and validated successfully")


 ## Diagnostics - Optional - Compare total LH size - Frozen cell

Test for analysis and troubleshooting.


In [ ]:
# Compare total size of source and backup lakehouses
print("Comparing lakehouse sizes...")
print("=" * 80)

# Get source tables
source_tables = lake.get_lakehouse_tables(lakehouse=source_lakehouse_name, workspace=source_workspace_id)

# Get backup tables
backup_tables = lake.get_lakehouse_tables(lakehouse=backup_lakehouse_name, workspace=backup_workspace_id)

# Get size details for each table using DESCRIBE DETAIL
source_sizes = []
for table_name in source_tables['Table Name']:
    try:
        path = f"abfss://{source_workspace_id}@onelake.dfs.fabric.microsoft.com/{source_lakehouse_id}/Tables/{table_name}"
        detail = spark.sql(f"DESCRIBE DETAIL delta.`{path}`").collect()[0]
        source_sizes.append({
            'table_name': table_name,
            'size_bytes': detail['sizeInBytes'],
            'num_files': detail['numFiles']
        })
    except:
        pass

backup_sizes = []
for table_name in backup_tables['Table Name']:
    try:
        path = f"abfss://{backup_workspace_id}@onelake.dfs.fabric.microsoft.com/{backup_lakehouse_id}/Tables/{table_name}"
        detail = spark.sql(f"DESCRIBE DETAIL delta.`{path}`").collect()[0]
        backup_sizes.append({
            'table_name': table_name,
            'size_bytes': detail['sizeInBytes'],
            'num_files': detail['numFiles']
        })
    except:
        pass

# Calculate totals
source_total_bytes = sum(s['size_bytes'] for s in source_sizes)
source_total_mb = source_total_bytes / (1024 * 1024)
backup_total_bytes = sum(s['size_bytes'] for s in backup_sizes)
backup_total_mb = backup_total_bytes / (1024 * 1024)

# Display comparison
comparison_data = {
    'Lakehouse': ['Source', 'Backup'],
    'Table_Count': [len(source_sizes), len(backup_sizes)],
    'Total_Size_MB': [source_total_mb, backup_total_mb]
}
comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

# Size difference
size_diff_mb = backup_total_mb - source_total_mb
size_pct = (backup_total_mb / source_total_mb * 100) if source_total_mb > 0 else 0

print(f"\nSize Comparison:")
print(f"Source:      {source_total_mb:,.2f} MB ({len(source_sizes)} tables)")
print(f"Backup:      {backup_total_mb:,.2f} MB ({len(backup_sizes)} tables)")
print(f"Difference:  {size_diff_mb:+,.2f} MB ({size_pct:.1f}%)")

# Identify missing tables if any
source_table_names = set(source_tables['Table Name'])
backup_table_names = set(backup_tables['Table Name'])

missing_in_backup = source_table_names - backup_table_names
missing_in_source = backup_table_names - source_table_names

if missing_in_backup:
    print(f"\nTables in Source but NOT in Backup ({len(missing_in_backup)}):")
    for table in missing_in_backup:
        print(f"  - {table}")

if missing_in_source:
    print(f"\nTables in Backup but NOT in Source ({len(missing_in_source)}):")
    for table in missing_in_source:
        print(f"  - {table}")

if not missing_in_backup and not missing_in_source:
    print("\n✓ All tables present in both lakehouses")